In [1]:
# NORTHSTAR — Solace Browser: App Ecosystem QA (app-store, app-detail, create-app)
# DNA: app_qa = structure(HTML5) x store(search+categories+cards) x detail(run+schedule+evidence) x create(4-step wizard) x i18n
# Template: solace-cli/notebooks/qa-templates/template-page-qa.ipynb
# Towers: T1(Masters) T7(Kids) T8(Elders) T15(App Makers) T23(Web)
# Auth: 65537 | Papers: 46,47,49,50
#
# File-based probes — reads HTML/JS/CSS files directly (no running server needed)
# REAL assertions. No mocks. No stubs.

import json, re, hashlib
from pathlib import Path
from datetime import datetime

NORTHSTAR = "solace-browser-app-ecosystem-qa"
NOTEBOOK_PATH = "notebooks/qa/26-app-ecosystem-qa.ipynb"
PROJECT = "solace-browser"
MIN_SCORE = 70

BROWSER = Path("/home/phuc/projects/solace-browser")
WEB = BROWSER / "web"

# App ecosystem pages under test
APP_PAGES = ["app-store", "app-detail", "create-app"]

results = []

def record(probe_id, name, passed, detail="", tower_ref=""):
    status = "PASS" if passed else "FAIL"
    results.append({"id": probe_id, "name": name, "status": status,
                    "detail": detail, "tower_ref": tower_ref})
    print(f"  {status}: {name}" + (f" -- {detail}" if detail and not passed else ""))

def read_page(name):
    return (WEB / f"{name}.html").read_text(encoding="utf-8")

# Verify all pages exist
for page in APP_PAGES:
    path = WEB / f"{page}.html"
    assert path.exists(), f"{page}.html missing at {path}"

print(f"BOOTSTRAP: {len(APP_PAGES)} app ecosystem pages under test")
print(f"Pages: {APP_PAGES}")

BOOTSTRAP: 3 app ecosystem pages under test
Pages: ['app-store', 'app-detail', 'create-app']


In [2]:
# P1: HTML Structure — all 3 pages have proper HTML5 structure
print("=== P1: HTML Structure ===")

REQUIRED_STRUCTURE = {
    "doctype": r"<!DOCTYPE html>",
    "html_lang": r'<html\s+lang="[a-z]{2}"',
    "title": r"<title>.+</title>",
    "meta_charset": r'<meta\s+charset="[uU][tT][fF]-8"',
    "meta_viewport": r'<meta\s+name="viewport"',
    "csp_header": r'Content-Security-Policy',
}

for page in APP_PAGES:
    html = read_page(page)
    for check_name, pattern in REQUIRED_STRUCTURE.items():
        found = bool(re.search(pattern, html, re.IGNORECASE))
        record(f"P1-{page}-{check_name}", f"{page}.html has {check_name}", found,
               detail=f"Pattern {pattern!r} not found in {page}.html" if not found else "",
               tower_ref="T1,T15,T23")

    # layout.js loaded for shared nav
    has_layout = bool(re.search(r'layout\.js', html))
    record(f"P1-{page}-layout_js", f"{page}.html loads layout.js", has_layout,
           detail="layout.js not loaded" if not has_layout else "", tower_ref="T1,T23")

    # header-slot present
    has_header = bool(re.search(r'id="header-slot"', html))
    record(f"P1-{page}-header_slot", f"{page}.html has header-slot", has_header,
           detail="Missing #header-slot" if not has_header else "", tower_ref="T1,T23")

print(f"\nP1 complete: {sum(1 for r in results if r['id'].startswith('P1') and r['status']=='PASS')}/{sum(1 for r in results if r['id'].startswith('P1'))} passed")

=== P1: HTML Structure ===
  PASS: app-store.html has doctype
  PASS: app-store.html has html_lang
  PASS: app-store.html has title
  PASS: app-store.html has meta_charset
  PASS: app-store.html has meta_viewport
  PASS: app-store.html has csp_header
  PASS: app-store.html loads layout.js
  PASS: app-store.html has header-slot
  PASS: app-detail.html has doctype
  PASS: app-detail.html has html_lang
  PASS: app-detail.html has title
  PASS: app-detail.html has meta_charset
  PASS: app-detail.html has meta_viewport
  PASS: app-detail.html has csp_header
  PASS: app-detail.html loads layout.js
  PASS: app-detail.html has header-slot
  PASS: create-app.html has doctype
  PASS: create-app.html has html_lang
  PASS: create-app.html has title
  PASS: create-app.html has meta_charset
  PASS: create-app.html has meta_viewport
  PASS: create-app.html has csp_header
  PASS: create-app.html loads layout.js
  PASS: create-app.html has header-slot

P1 complete: 24/24 passed


In [3]:
# P2: App Store — has search input, categories section, app cards, proposal form
print("=== P2: App Store Content ===")

store_html = read_page("app-store")

store_checks = {
    "search_input": (r'id="app-search"', "Search input field"),
    "search_label": (r'class="sr-only"[^>]*>.*[Ss]earch apps', "Accessible search label"),
    "categories_section": (r'id="app-store-categories"', "Categories container"),
    "categories_heading": (r'appstore_categories_h2|Communications.*Productivity', "Category headings"),
    "no_api_exclusives": (r'No-API Exclusives|appstore_noapi_h2', "No-API exclusives section"),
    "propose_form": (r'id="app-proposal-form"', "App proposal form"),
    "how_it_works": (r'appstore_how_h2|How every app works', "How it works section"),
    "create_app_link": (r'href="[^"]*create-app"', "Link to create-app page"),
    "fun_packs": (r'id="fun-pack-grid"|Fun.*Personality', "Fun packs section"),
}

for name, (pattern, desc) in store_checks.items():
    found = bool(re.search(pattern, store_html))
    record(f"P2-store-{name}", f"app-store.html has {desc}", found,
           detail=f"Missing: {desc}" if not found else "", tower_ref="T15,T23")

print(f"\nP2 complete: {sum(1 for r in results if r['id'].startswith('P2') and r['status']=='PASS')}/{sum(1 for r in results if r['id'].startswith('P2'))} passed")

=== P2: App Store Content ===
  PASS: app-store.html has Search input field
  PASS: app-store.html has Accessible search label
  PASS: app-store.html has Categories container
  PASS: app-store.html has Category headings
  PASS: app-store.html has No-API exclusives section
  PASS: app-store.html has App proposal form
  PASS: app-store.html has How it works section
  PASS: app-store.html has Link to create-app page
  PASS: app-store.html has Fun packs section

P2 complete: 9/9 passed


In [4]:
# P3: App Detail — has run button, schedule link, evidence section, tabs
print("=== P3: App Detail Content ===")

detail_html = read_page("app-detail")

detail_checks = {
    "run_button": (r'id="btnRun"|Run Now', "Run Now button"),
    "schedule_link": (r'class="ad-schedule-btn"|href="[^"]*schedule"', "Schedule link/button"),
    "evidence_section": (r'ad-result-evidence|Evidence|evidence', "Evidence display section"),
    "tabs": (r'class="sb-tab"', "Tab navigation"),
    "runs_tab": (r'data-tab="tab-runs"|tab_runs', "Runs tab"),
    "back_link": (r'class="ad-back"', "Back to app-store link"),
    "activate_button": (r'id="btnActivate"', "Activate button"),
    "compliance_tab": (r'Compliance|eSign|Part 11', "Compliance/Evidence tab"),
}

for name, (pattern, desc) in detail_checks.items():
    found = bool(re.search(pattern, detail_html))
    record(f"P3-detail-{name}", f"app-detail.html has {desc}", found,
           detail=f"Missing: {desc}" if not found else "", tower_ref="T15,T23")

# Check runs table structure
has_runs_table = bool(re.search(r'id="runsTable"', detail_html))
record("P3-detail-runs_table", "app-detail.html has runs history table", has_runs_table,
       detail="Missing runs history table" if not has_runs_table else "", tower_ref="T15,T23")

# Check that detail page has app configuration fields
has_config = bool(re.search(r'schedule.*label|prompt.*label|frequency', detail_html, re.IGNORECASE))
record("P3-detail-config_fields", "app-detail.html has configuration fields", has_config,
       detail="Missing app config fields" if not has_config else "", tower_ref="T15,T23")

print(f"\nP3 complete: {sum(1 for r in results if r['id'].startswith('P3') and r['status']=='PASS')}/{sum(1 for r in results if r['id'].startswith('P3'))} passed")

=== P3: App Detail Content ===
  PASS: app-detail.html has Run Now button
  PASS: app-detail.html has Schedule link/button
  PASS: app-detail.html has Evidence display section
  PASS: app-detail.html has Tab navigation
  PASS: app-detail.html has Runs tab
  PASS: app-detail.html has Back to app-store link
  PASS: app-detail.html has Activate button
  PASS: app-detail.html has Compliance/Evidence tab
  PASS: app-detail.html has runs history table
  PASS: app-detail.html has configuration fields

P3 complete: 10/10 passed


In [5]:
# P4: Create App — has 4-step wizard (Describe, Preview, Test, Submit)
print("=== P4: Create App Wizard ===")

create_html = read_page("create-app")

# Check all 4 wizard steps exist
wizard_steps = {
    "step1_describe": (r'Step 1.*Describe|ca_step1_kicker', "Step 1 — Describe"),
    "step2_preview": (r'Step 2.*Preview|ca_step2_kicker', "Step 2 — Preview"),
    "step3_test": (r'Step 3.*Test|ca_step3_kicker', "Step 3 — Test"),
    "step4_submit": (r'Step 4.*Submit|ca_step4_kicker', "Step 4 — Submit"),
}

for name, (pattern, desc) in wizard_steps.items():
    found = bool(re.search(pattern, create_html))
    record(f"P4-create-{name}", f"create-app.html has {desc}", found,
           detail=f"Missing wizard step: {desc}" if not found else "", tower_ref="T15,T23")

# Check progress indicator (step dots)
has_progress = bool(re.search(r'class="ca-progress"|ca-dot', create_html))
record("P4-create-progress_dots", "create-app.html has step progress indicator", has_progress,
       detail="Missing progress dots" if not has_progress else "", tower_ref="T15,T23")

# Check hero section with YinYang branding (class may include "reveal" modifier)
has_hero = bool(re.search(r'class="ca-hero', create_html))
record("P4-create-hero", "create-app.html has hero section", has_hero,
       detail="Missing hero section" if not has_hero else "", tower_ref="T15,T23")

# Check i18n support
i18n_count = len(re.findall(r'data-i18n="[^"]+"', create_html))
record("P4-create-i18n", f"create-app.html has data-i18n attributes ({i18n_count})",
       i18n_count >= 4,
       detail=f"Only {i18n_count} i18n attrs, expected >= 4" if i18n_count < 4 else "",
       tower_ref="T9,T15")

# Check navigation to app-store: either direct link OR layout.js provides shared nav header
has_direct_link = bool(re.search(r'href="[^"]*app-store"', create_html))
has_layout_nav = bool(re.search(r'layout\.js', create_html))
record("P4-create-store_link", "create-app.html has navigation to app-store (direct or via layout.js header)",
       has_direct_link or has_layout_nav,
       detail="No link to app-store and no layout.js for shared nav" if not (has_direct_link or has_layout_nav) else "",
       tower_ref="T15,T23")

print(f"\nP4 complete: {sum(1 for r in results if r['id'].startswith('P4') and r['status']=='PASS')}/{sum(1 for r in results if r['id'].startswith('P4'))} passed")

=== P4: Create App Wizard ===
  PASS: create-app.html has Step 1 — Describe
  PASS: create-app.html has Step 2 — Preview
  PASS: create-app.html has Step 3 — Test
  PASS: create-app.html has Step 4 — Submit
  PASS: create-app.html has step progress indicator
  PASS: create-app.html has hero section
  PASS: create-app.html has data-i18n attributes (17)
  PASS: create-app.html has navigation to app-store (direct or via layout.js header)

P4 complete: 8/8 passed


In [6]:
# P5: Evidence Summary
print("=== P5: Evidence Summary ===\n")

total = len(results)
passed = sum(1 for r in results if r["status"] == "PASS")
failed = total - passed
score = round((passed / total) * 100, 1) if total > 0 else 0.0

evidence_blob = json.dumps({
    "notebook": "26-app-ecosystem-qa",
    "timestamp": datetime.now().isoformat(),
    "total": total,
    "passed": passed,
    "failed": failed,
    "score": score,
    "results": results,
}, indent=2)

evidence_hash = hashlib.sha256(evidence_blob.encode()).hexdigest()

print(f"  Total probes : {total}")
print(f"  Passed       : {passed}")
print(f"  Failed       : {failed}")
print(f"  Score        : {score}%")
print(f"  Evidence hash: {evidence_hash[:16]}...")
print()

if failed > 0:
    print("FAILURES:")
    for r in results:
        if r["status"] == "FAIL":
            print(f"  FAIL: {r['name']} -- {r['detail']}")

assert score >= MIN_SCORE, f"Score {score}% below minimum {MIN_SCORE}%"
print(f"\nVERDICT: PASS ({score}% >= {MIN_SCORE}% minimum)")

=== P5: Evidence Summary ===

  Total probes : 51
  Passed       : 51
  Failed       : 0
  Score        : 100.0%
  Evidence hash: 384b7f0a62908f5a...


VERDICT: PASS (100.0% >= 70% minimum)
